In [1]:
# Setup: load config and prepare datasets using the library helpers
from pathlib import Path

import pandas as pd
import xarray as xr
from IPython.display import Image, display

from swissclim_evaluations.cli import _load_yaml, prepare_datasets, run_selected
from swissclim_evaluations.metrics.probabilistic import (
    run_probabilistic_wbx,
)

# Locate configuration relative to the notebook location
cfg_path = None
for base in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    candidate = base / "config" / "example_config.yaml"
    if candidate.is_file():
        cfg_path = candidate
        break
if cfg_path is None:
    raise FileNotFoundError(
        "Could not find config/example_config.yaml in cwd, parent, or grandparent directories."
    )

# Load configuration and derive output root relative to project root (parent of notebooks)
cfg = _load_yaml(cfg_path)
project_root = Path.cwd().parent
cfg_output = cfg.get("paths", {}).get("output_root", "output/notebook_prob_wbx")
out_root = (
    (project_root / cfg_output).resolve() if not cfg_output.startswith("/") else Path(cfg_output)
)
out_root.mkdir(parents=True, exist_ok=True)
print(f"Output root: {out_root}")

# Prepare datasets once
(
    ds_targets,
    ds_predictions,
    ds_targets_std,
    ds_predictions_std,
) = prepare_datasets(cfg)

Output root: /capstor/store/cscs/swissai/a122/sadamov/SwissClim_Evaluations/output/verification_esfm


/capstor/store/cscs/swissai/a122/sadamov/SwissClim_Evaluations/src/swissclim_evaluations/data.py:101: RuntimeWarning: Rechunking ml to policy {'init_time': 1, 'lead_time': 1, 'level': 1, 'ensemble': -1, 'latitude': -1, 'longitude': -1}. This may increase memory usage and runtime.
  warnings.warn(


In [2]:
# Toggle to run the full pipeline via CLI orchestration
RUN_ALL = False

In [3]:
# If RUN_ALL is enabled, execute the orchestrator using the same config
# and skip the per-chapter call below to avoid duplication.
if RUN_ALL:
    run_selected(cfg)

In [4]:
# Show the prepared datasets (quick peek)
ds_targets

<xarray.Dataset> Size: 112MB
Dimensions:                  (init_time: 27, lead_time: 1, latitude: 720,
                              longitude: 1440)
Coordinates:
  * init_time                (init_time) datetime64[ns] 216B 2023-01-02 ... 2...
  * lead_time                (lead_time) timedelta64[ns] 8B 06:00:00
  * latitude                 (latitude) float32 3kB 90.0 89.75 ... -89.5 -89.75
  * longitude                (longitude) float32 6kB 0.0 0.25 ... 359.5 359.8
    valid_time               (init_time, lead_time) datetime64[ns] 216B 2023-...
Data variables:
    10m_u_component_of_wind  (latitude, longitude, init_time, lead_time) float32 112MB dask.array<chunksize=(720, 1440, 1, 1), meta=np.ndarray>
Attributes:
    last_updated:      2024-10-17 20:04:10.783634
    valid_time_start:  1940-01-01
    valid_time_stop:   2024-07-31

In [5]:
# Show predictions (quick peek)
ds_predictions

<xarray.Dataset> Size: 896MB
Dimensions:                  (init_time: 27, lead_time: 1, ensemble: 8,
                              latitude: 720, longitude: 1440)
Coordinates:
  * init_time                (init_time) datetime64[ns] 216B 2023-01-02 ... 2...
  * lead_time                (lead_time) timedelta64[ns] 8B 06:00:00
  * ensemble                 (ensemble) int64 64B 0 1 2 3 4 5 6 7
  * latitude                 (latitude) float32 3kB 90.0 89.75 ... -89.5 -89.75
  * longitude                (longitude) float32 6kB 0.0 0.25 ... 359.5 359.8
    valid_time               (init_time, lead_time) datetime64[ns] 216B dask.array<chunksize=(1, 1), meta=np.ndarray>
Data variables:
    10m_u_component_of_wind  (ensemble, latitude, longitude, init_time, lead_time) float32 896MB dask.array<chunksize=(8, 720, 1440, 1, 1), meta=np.ndarray>
Attributes:
    model:    model_ckpt-step=7300-loss_train=0.07.ckpt

In [6]:
ds_targets

<xarray.Dataset> Size: 112MB
Dimensions:                  (init_time: 27, lead_time: 1, latitude: 720,
                              longitude: 1440)
Coordinates:
  * init_time                (init_time) datetime64[ns] 216B 2023-01-02 ... 2...
  * lead_time                (lead_time) timedelta64[ns] 8B 06:00:00
  * latitude                 (latitude) float32 3kB 90.0 89.75 ... -89.5 -89.75
  * longitude                (longitude) float32 6kB 0.0 0.25 ... 359.5 359.8
    valid_time               (init_time, lead_time) datetime64[ns] 216B 2023-...
Data variables:
    10m_u_component_of_wind  (latitude, longitude, init_time, lead_time) float32 112MB dask.array<chunksize=(720, 1440, 1, 1), meta=np.ndarray>
Attributes:
    last_updated:      2024-10-17 20:04:10.783634
    valid_time_start:  1940-01-01
    valid_time_stop:   2024-07-31

In [7]:
ds_predictions

<xarray.Dataset> Size: 896MB
Dimensions:                  (init_time: 27, lead_time: 1, ensemble: 8,
                              latitude: 720, longitude: 1440)
Coordinates:
  * init_time                (init_time) datetime64[ns] 216B 2023-01-02 ... 2...
  * lead_time                (lead_time) timedelta64[ns] 8B 06:00:00
  * ensemble                 (ensemble) int64 64B 0 1 2 3 4 5 6 7
  * latitude                 (latitude) float32 3kB 90.0 89.75 ... -89.5 -89.75
  * longitude                (longitude) float32 6kB 0.0 0.25 ... 359.5 359.8
    valid_time               (init_time, lead_time) datetime64[ns] 216B dask.array<chunksize=(1, 1), meta=np.ndarray>
Data variables:
    10m_u_component_of_wind  (ensemble, latitude, longitude, init_time, lead_time) float32 896MB dask.array<chunksize=(8, 720, 1440, 1, 1), meta=np.ndarray>
Attributes:
    model:    model_ckpt-step=7300-loss_train=0.07.ckpt

In [8]:
# Compute standardized WBX outputs (CSVs, NetCDFs, optional CRPS map) via library function
if not RUN_ALL:
    run_probabilistic_wbx(
        ds_target=ds_targets,
        ds_prediction=ds_predictions,
        out_root=out_root,
        plotting_cfg=cfg.get("plotting", {}),
        all_cfg=cfg,
    )

[probabilistic] saved /capstor/store/cscs/swissai/a122/sadamov/SwissClim_Evaluations/output/verification_esfm/probabilistic/spread_skill_ratio.csv
[probabilistic] saved /capstor/store/cscs/swissai/a122/sadamov/SwissClim_Evaluations/output/verification_esfm/probabilistic/crps_ensemble.csv
Wrote: /capstor/store/cscs/swissai/a122/sadamov/SwissClim_Evaluations/output/verification_esfm/probabilistic/probabilistic_metrics_temporal.nc
Wrote: /capstor/store/cscs/swissai/a122/sadamov/SwissClim_Evaluations/output/verification_esfm/probabilistic/probabilistic_metrics_spatial.nc
[probabilistic] saved /capstor/store/cscs/swissai/a122/sadamov/SwissClim_Evaluations/output/verification_esfm/probabilistic/crps_map_wbx_10m_u_component_of_wind.png


In [9]:
# Read temporal results from disk using resolved out_root
section = out_root / "probabilistic_wbx"
temporal_fn = section / "probabilistic_metrics_temporal.nc"
if temporal_fn.exists():
    ds_temporal = xr.load_dataset(temporal_fn, engine="scipy")
    print("Loaded:", temporal_fn)
    # Show a quick summary: dims, variables, and first few values for one variable
    print("dims:", ds_temporal.dims)
    print("data_vars:", list(ds_temporal.data_vars)[:6], "...")
    # If CRPS exists, show a small table grouped over time
    crps_vars = [v for v in ds_temporal.data_vars if v.startswith("CRPS.")]
    if crps_vars:
        try:
            preview = ds_temporal[crps_vars[0]].mean(
                [d for d in ["init_time", "lead_time"] if d in ds_temporal.dims]
            )
            display(preview.to_dataframe().head(10))
        except Exception:
            pass
else:
    print(f"Temporal file not found: {temporal_fn}")

Temporal file not found: /capstor/store/cscs/swissai/a122/sadamov/SwissClim_Evaluations/output/verification_esfm/probabilistic_wbx/probabilistic_metrics_temporal.nc


In [10]:
# List generated outputs for this module (no recomputation)
section = Path(cfg["paths"]["output_root"]) / "probabilistic_wbx"
if section.exists():
    for p in sorted(section.glob("**/*")):
        print(p)
else:
    print(f"No outputs yet at {section}. Run the cells above.")

No outputs yet at output/verification_esfm/probabilistic_wbx. Run the cells above.


In [11]:
# Read spatial results from disk
section = Path(cfg["paths"]["output_root"]) / "probabilistic_wbx"
spatial_fn = section / "probabilistic_metrics_spatial.nc"
if spatial_fn.exists():
    ds_spatial = xr.load_dataset(spatial_fn, engine="scipy")
    print("Loaded:", spatial_fn)
    print("dims:", ds_spatial.dims)
    print("data_vars:", list(ds_spatial.data_vars)[:6], "...")
    # Show a simple regional preview if available
    region_dim = "region" if "region" in ds_spatial.dims else None
    crps_vars = [v for v in ds_spatial.data_vars if v.startswith("CRPS.")]
    if region_dim and crps_vars:
        try:
            preview = ds_spatial[crps_vars[0]].to_dataframe().groupby("region").mean().head(10)
            display(preview)
        except Exception:
            pass
else:
    print(f"Spatial file not found: {spatial_fn}")

Spatial file not found: output/verification_esfm/probabilistic_wbx/probabilistic_metrics_spatial.nc


In [12]:
# Read CSV summaries (SSR and CRPS ensemble)
section = Path(cfg["paths"]["output_root"]) / "probabilistic_wbx"
csv_files = ["spread_skill_ratio.csv", "crps_ensemble.csv"]
for name in csv_files:
    p = section / name
    if p.exists():
        try:
            df = pd.read_csv(p)
            print("Loaded:", p)
            display(df.head(10))
        except Exception as e:
            print(f"Failed reading {p}: {e}")
    else:
        print(f"Not found: {p}")

Not found: output/verification_esfm/probabilistic_wbx/spread_skill_ratio.csv
Not found: output/verification_esfm/probabilistic_wbx/crps_ensemble.csv


In [13]:
# Display a CRPS map PNG if present
section = Path(cfg["paths"]["output_root"]) / "probabilistic_wbx"
pngs = sorted(section.glob("crps_map_*.png"))
if pngs:
    print("Found:", pngs[0])
    display(Image(filename=str(pngs[0])))
else:
    print("No CRPS map PNGs found.")

No CRPS map PNGs found.
